# Chebyshev approximation for the nozzle inverse

Compare reciprocal fits on the original singular-value interval and prepare the exact and bounded approximate LCU probabilities with FlagQuantum.

Use the local FlagQuantum workflow.

In [ ]:
from pathlib import Path

from IPython.display import Image, display

from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview import (
    model,
    quantum,
    workflow,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.main import (
    print_report,
)
from examples.synthesis.applications.cfd.qls_for_hybrid_solvers.chebyshev_approximation.overview.plots import (
    save_figures,
)

Choose the output directory for the plots.

In [ ]:
figures = Path("notebook-figures")

Compare both error metrics on the original singular-value interval.

In [ ]:
print(model.APPROXIMATIONS)

Restore the full original nozzle matrix and compute the banded scaling.

In [ ]:
matrix = model.source_matrix()
nozzle = model.build_model()

Inspect the dense matrix used for the singular-value calculation.

In [ ]:
print("Matrix shape:", tuple(matrix.shape))
print("Nonzero entries:", int(matrix.count_nonzero()))

Measure the scaled singular values and keep the original polynomial degrees.

In [ ]:
print("Banded normalization:", nozzle.alpha)
print("Singular value interval:", nozzle.smallest, nozzle.largest)

Generate the four polynomials and compute their errors.

In [ ]:
cases = workflow.approximations(nozzle)
for case in cases:
    print(case.method, case.degree, case.relative_error, case.uniform_error)

Fit all 128 inverse LCU coefficients, then prepare both distributions.

In [ ]:
fitted = model.fit_alternating(cases[1].coefficients)
exact, approximate = quantum.compare(fitted)
result = workflow.Result(nozzle, cases, fitted, exact, approximate)
print_report(result)

Check that the approximate circuit uses less than half as many RY gates.

In [ ]:
assert approximate.resources.operation_counts["ry"] * 2 < exact.resources.operation_counts["ry"]

Plot the reciprocal approximations and measured coefficient distributions.

In [ ]:
save_figures(result, figures)
display(Image(filename=str(figures / "reciprocal-approximations.png")))
display(Image(filename=str(figures / "prepared-coefficients.png")))